### AUC diaria: DQN frente a no actuar (A1)

Este notebook usa únicamente `daily_metrics.csv` de la carpeta de outputs de v3.
Compara, en los cuatro escenarios de prueba, la AUC diaria sobre pacientes
reservados de dos políticas aplicadas al mismo replay:

- **DQN:** política greedy seleccionada en validación.
- **A1:** deferir siempre; el pipeline inicial nunca se actualiza.

La diferencia entre ambas curvas muestra cuánto AUC recupera el agente frente a
no hacer mantenimiento. Las líneas verticales marcan el inicio de cada etapa de
cambio (M = medición, C = relación features–etiqueta), información oculta al agente.

### Bibliotecas y ruta de datos

El notebook se ejecuta desde `kaggle/dqn-dummy/v3/outputs/`; si se ejecuta desde
la raíz del repositorio, se resuelve la misma carpeta.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

candidates = [Path('.'), Path('kaggle/dqn-dummy/v3/outputs')]
OUT = next(p for p in candidates if (p / 'daily_metrics.csv').is_file())
(OUT / 'figures').mkdir(exist_ok=True)
SCENARIOS = ['stable', 'covariate', 'concept', 'combined']
STYLE = {'dqn': {'color': '#1f6f9e', 'ls': '-', 'label': 'DQN'},
         'A1': {'color': '#b3741f', 'ls': '--', 'label': 'A1 (sin acciones)'}}
print('Outputs:', OUT.resolve())

### Cargar y validar métricas diarias

Se conservan solo las políticas DQN y A1. Cada escenario debe tener 16 días por
política y la AUC debe estar en [0, 1].

In [ ]:
daily = pd.read_csv(OUT / 'daily_metrics.csv')
data = daily.loc[daily.policy.isin(['dqn', 'A1']), ['scenario', 'policy', 'day', 'auc', 'cov_stage', 'concept_stage']]
counts = data.groupby(['scenario', 'policy']).size()
assert set(data.scenario) == set(SCENARIOS)
assert counts.eq(counts.iloc[0]).all(), counts
assert data.auc.dropna().between(0, 1).all()
print(data.shape, '| días por escenario y política:', int(counts.iloc[0]))
data.head()

### Gráfico: AUC diaria por escenario

Un panel por escenario con la misma escala vertical. La línea gris punteada es
el umbral operativo AUC = 0,85 usado en la observación del agente.

In [ ]:
def stage_starts(table, column):
    """Return days where a hidden stage begins, with its stage number."""
    t = table.drop_duplicates('day').sort_values('day')
    change = t[column].ne(t[column].shift(fill_value=0)) & t[column].gt(0)
    return list(zip(t.loc[change, 'day'], t.loc[change, column]))

plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
fig, axes = plt.subplots(2, 2, figsize=(16, 9), sharex=True, sharey=True)
for ax, scenario in zip(axes.flat, SCENARIOS):
    subset = data.loc[data.scenario.eq(scenario)]
    for column, prefix in (('cov_stage', 'M'), ('concept_stage', 'C')):
        for day, stage in stage_starts(subset, column):
            ax.axvline(day, color='#b8bcc2', lw=1, zorder=0)
            ax.text(day + .1, 1.0, f'{prefix}{int(stage)}', fontsize=9, color='#5f6670', va='top')
    ax.axhline(.85, color='#9aa0a6', ls=':', lw=1, zorder=0)
    ends = {}
    for policy in ('dqn', 'A1'):
        s = subset.loc[subset.policy.eq(policy)].sort_values('day')
        style = STYLE[policy]
        ax.plot(s.day, s.auc, ls=style['ls'], color=style['color'], lw=2, marker='o', ms=4, label=style['label'])
        ends[policy] = s.dropna(subset=['auc']).iloc[-1]
    # Final-value labels; merged when both endpoints would overlap.
    if abs(ends['dqn'].auc - ends['A1'].auc) < .03:
        labels = [(f"DQN = A1 {ends['dqn'].auc:.2f}", ends['dqn'])]
    else:
        labels = [(f"{STYLE[p]['label'].split(' ')[0]} {ends[p].auc:.2f}", ends[p]) for p in ('dqn', 'A1')]
    for text, last in labels:
        ax.annotate(text, (last.day, last.auc), xytext=(6, 0), textcoords='offset points',
                    va='center', fontsize=9, color='#3c4043')
    mean_dqn = subset.loc[subset.policy.eq('dqn'), 'auc'].mean()
    mean_a1 = subset.loc[subset.policy.eq('A1'), 'auc'].mean()
    ax.set_title(f'{scenario}  ·  AUC media DQN {mean_dqn:.3f} vs A1 {mean_a1:.3f}', fontsize=11)
    ax.set_ylim(.45, 1.02)
    ax.set_xlim(.5, 17.5)
    ax.grid(axis='y', color='#eceef0', lw=.8)
for ax in axes[1]:
    ax.set_xlabel('Día simulado')
for ax in axes[:, 0]:
    ax.set_ylabel('AUC día actual (pacientes reservados)')
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(.5, 1.0))
fig.suptitle('AUC diaria en prueba: DQN frente a no actuar (A1)', y=1.04, fontsize=14)
fig.tight_layout()
for ext in ('png', 'svg'):
    fig.savefig(OUT / 'figures' / f'10_auc_dqn_vs_a1.{ext}', dpi=160 if ext == 'png' else None, bbox_inches='tight')
plt.show()

### Tabla de soporte

AUC media por escenario, separando días sin cambio (antes de la primera etapa)
y días con cambio. `ganancia` = AUC DQN − AUC A1.

In [ ]:
data['periodo'] = np.where(data.cov_stage.gt(0) | data.concept_stage.gt(0), 'con cambio', 'sin cambio')
table = (data.pivot_table(index=['scenario', 'periodo'], columns='policy', values='auc', aggfunc='mean')
             .rename(columns={'dqn': 'DQN'})[['DQN', 'A1']])
table['ganancia'] = table.DQN - table.A1
table = table.reindex(SCENARIOS, level=0).round(3)
table.to_csv(OUT / 'auc_dqn_vs_a1.csv')
table